In [ ]:
## Analysis of inflow outputs for renal cell carcinoma dataset
## Dr Daniyal Jafree, Lotfollahi Group, CellGen Programme, Wellcome Sanger Institute
## Final pertubation experiment, remove macrophages from TLS and predicts effects on TLS T cell subsets
## Final Version - 20th July 2025

In [ ]:
# Load packages

import numpy as np
import pickle
import scanpy as sc
import squidpy as sq
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
import glasbey
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from pathlib import Path
import os, sys
from scipy.sparse import issparse
import matplotlib.pyplot as plt
import anndata
import gseapy as gp
for path in list_pathstoadd:
    if(path not in sys.path):
        sys.path.append(path)

In [ ]:
# Setup for pertubation (old script, please see new API)

t_begin = time.time()
list_varnames_todump = [
    'x_spl_original',
    'x_spl_modified',
    'np_NCC_original',
    'np_NCC_modified',
    'adata_selected_region_original',
    'adata_selected_region_modified',
    'list_bool_idx_slice_orig_region'
]
for varname in list_varnames_todump:
    if varname[0:len('adata_')] == 'adata_':
        # globals()[varname].write_h5ad(
        #     "NonGit/June11th_for_Daniyal/{}.h5ad".format(varname)
        # )
        exec(
            "{} = sc.read_h5ad('/home/jupyter/Final_pertubation_RCC/{}.h5ad')".format(
                varname,
                varname
            )
        )
        print("Loaded {}".format(varname))
    else:
        with open("/home/jupyter/Final_pertubation_RCC/{}.pkl".format(varname), 'rb') as f:
            exec(
                '{} = pickle.load(f)'.format(
                    varname
                )
            )
            print("Loaded {}".format(varname))


print("------ Took {} seconds.".format(
    time.time() - t_begin
))

In [ ]:
# settings

n_genes_showranking = 20

colpal = glasbey.create_palette(
    palette_size=len(set(adata_selected_region_original.obs['inflow_cell_type']))
)
sns.palplot(colpal)

kwargs_plotgraph_before_modification = {
    'edges_width':0.1,
    'legend_fontsize':14,
    'figsize':[10,10],
    'size':10.0
}
# local settings (TODO:modify if needed) ===
kwargs_plotgraph_after_modification = {
    'edges_width':0.1,
    'legend_fontsize':14,
    'figsize':[10,10],
    'size':2.0
}

union_all_CT = set(
    adata_selected_region_original.obs['inflow_cell_type'],
).union(
    set(adata_selected_region_modified.obs['inflow_cell_type'])
)
#num_generated_realisations = len(dict_original_varname_to_all_realisations['x_int'])

In [ ]:
### GENERATES FIGURES 2E

sq.pl.spatial_scatter(
    adata_selected_region_original,
    spatial_key='spatial',
    img=False,
    connectivity_key="spatial_connectivities",
    library_id='connectivities_key', #'connectivities_key',
    crop_coord=None,
    color=['inflow_cell_type'],
    palette=ListedColormap(colpal),
    **kwargs_plotgraph_before_modification
)
plt.show()

In [ ]:
### GENERATES FIGURES 2E

sq.pl.spatial_scatter(
    adata_selected_region_modified,
    spatial_key='spatial',
    img=False,
    connectivity_key="spatial_connectivities",
    library_id='connectivities_key', #'connectivities_key',
    crop_coord=None,
    color=['inflow_cell_type'],
    palette=ListedColormap(colpal),
    **kwargs_plotgraph_before_modification
)
plt.show()

In [ ]:
## Differential expression

# x_spl_original = np.stack(dict_original_varname_to_all_realisations['x_spl'], 0).mean(0)
# x_spl_modified = np.stack(dict_modified_varname_to_all_realisations['x_spl'], 0).mean(0)

for ct in union_all_CT: #set(adata.obs['inflow_cell_type']):
    if ct in set(adata_selected_region_original.obs['inflow_cell_type']):
        if ct in set(adata_selected_region_modified.obs['inflow_cell_type']):

            flag_skip = \
            (list(adata_selected_region_original.obs['inflow_cell_type']).count(ct) < 3) or \
            (list(adata_selected_region_modified.obs['inflow_cell_type']).count(ct) < 3) # local settings (TODO:modify if needed) ===

            if flag_skip:
                print("skipped for cell type '{}' due to small number of cells in the region.".format(ct))
            else:
                print("  Cell type {}, count in the original and modified version are {} and {}".format(
                        ct,
                        list(adata_selected_region_original.obs['inflow_cell_type']).count(ct),
                        list(adata_selected_region_modified.obs['inflow_cell_type']).count(ct)
                    ) + \
                    "\n   If these two numbers are too small, the differential analysis may produce unreliable results."
                )

                # gene scores to take into account ===
                # TODO_BASEVAL, TODO_WEIGHT_1, TODO_WEIGHT_2, TODO_WEIGHT_3 = 1.0, 0, 0, 0
                final_gene_score = 1.0

                # filter out based both on cell type and on if MCC is changed.
                '''
                dropping cells from orig was done via: 
                [list_bool_idx_slice_orig_region, :]
                which can be used to find cells whose MCC is changed.
                '''
                dict_map_idxmodif_to_idxorig = {
                    idx_modif:idx_orig
                    for idx_modif, idx_orig in enumerate(
                        np.where(np.array(list_bool_idx_slice_orig_region))[0].tolist()
                    )
                }
                dict_map_idxorig_to_idxmodif = {
                    idx_orig:idx_modif
                    for idx_modif, idx_orig in dict_map_idxmodif_to_idxorig.items()
                }
                list_selflag_MCCchanged_orig = [
                    bool(np.any(np_NCC_original[idx_orig] != np_NCC_modified[dict_map_idxorig_to_idxmodif[idx_orig]])) \
                    if(idx_orig in dict_map_idxorig_to_idxmodif.keys()) else False
                    for idx_orig in range(np_NCC_original.shape[0])
                ]
                list_selflag_MCCchanged_modif = [
                    bool(np.any(np_NCC_modified[idx_modif] != np_NCC_original[dict_map_idxmodif_to_idxorig[idx_modif]]))
                    for idx_modif in range(np_NCC_modified.shape[0])
                ]
                
                list_selflag_orig = np.logical_and(
                    np.array(adata_selected_region_original.obs['inflow_cell_type'] == ct),
                    np.array(list_selflag_MCCchanged_orig)
                ).tolist()  # filter based on both cell type and MCC
                list_selflag_modif = np.logical_and(
                    np.array(adata_selected_region_modified.obs['inflow_cell_type'] == ct),
                    np.array(list_selflag_MCCchanged_modif)
                ).tolist()  # filter based on both cell type and MCC

               

                
                
                adata_ct = sc.AnnData(
                    X=np.concatenate(
                        [x_spl_original[list_selflag_orig, :] + 0.0,
                         x_spl_modified[list_selflag_modif, :] + 0.0],
                        0
                    ) * final_gene_score,
                    obs=pd.DataFrame(
                        data=np.array(
                            [sum(list_selflag_orig)*['original'] +\
                             sum(list_selflag_modif)*['modified']]
                        ).T,
                        columns=['original_vs_modified']
                    ),
                    var=adata_selected_region_original.var
                )

                flag_skip = True
                if adata_ct.shape[0] >= 4:
                    if (int(adata_ct.obs['original_vs_modified'].value_counts()['original']) > 1) and (int(adata_ct.obs['original_vs_modified'].value_counts()['modified']) > 1):
                        flag_skip = False
                    

                if flag_skip:
                    print(">>>>>>>>>>>>>>>>>>>>\n>>>>>>>>>>>\nFor cell type `{}` all cells were filtered out --> no DE analysis.".format(ct))
                else:
                    
                
               
    
                    # IMPORTANT BUG:  normalize_total should not be called for the 2nd time on Xspl. 
                    #  Because the effect of size factor is already removed. 
                    #  sc.pp.normalize_total(adata_ct, target_sum=10000, inplace=True) # local settings (TODO:modify if needed) ===
                    adata_ct.layers['xspl_before_log1p'] = adata_ct.X.copy()
                    sc.pp.log1p(adata_ct) # local settings (TODO:modify if needed) ===
        
                    sc.tl.rank_genes_groups(
                        adata_ct,
                        'original_vs_modified',
                        method='wilcoxon',
                        n_genes=n_genes_showranking
                    )
                    print(adata_ct.uns['rank_genes_groups']['names'])
    
                    sc.pl.rank_genes_groups(adata_ct)
                    print(adata_ct.uns['rank_genes_groups']['names'])
                    
                    sc.pl.dotplot(
                        adata_ct,
                        var_names=\
                        [u[0] for u in adata_ct.uns['rank_genes_groups']['names'].tolist()]+\
                        [u[1] for u in adata_ct.uns['rank_genes_groups']['names'].tolist()],
                        groupby='original_vs_modified',
                        dendrogram=True,
                        title="Cell type: {}".format(ct),
                        size_title=20,
                        #cmap='jet_r',
                        layer='xspl_before_log1p',
                        mean_only_expressed=True,
                        standard_scale='var'
                    )
    
                    # # violin plots
                    # fig_num_rowcols = int(np.ceil(np.sqrt(
                    #     2 * len(adata_ct.uns['rank_genes_groups']['names'].tolist())
                    # )))
                    # plt.figure(figsize=[fig_num_rowcols*4, fig_num_rowcols*4])
                    # cnt_subfig = 1
                    # for g in [g1 for g1, _ in adata_ct.uns['rank_genes_groups']['names'].tolist()] +\
                    # [g2 for _, g2 in adata_ct.uns['rank_genes_groups']['names'].tolist()]:
                        
                    #     # get all original values
                    #     np_allvals_orig = np.concatenate([
                    #         xspl[list_selflag_orig]\
                    #          [:, adata_ct.var.index.tolist().index(g)].flatten() for xspl in  dict_original_varname_to_all_realisations['x_spl']
                    #     ])
                    #     assert np_allvals_orig.shape[0] == adata_ct.obs['original_vs_modified'].value_counts()['original'] * num_generated_realisations
    
                    #     # get all modified values
                    #     np_allvals_modif = np.concatenate([
                    #         xspl[list_selflag_modif]\
                    #          [:, adata_ct.var.index.tolist().index(g)].flatten() for xspl in  dict_modified_varname_to_all_realisations['x_spl']
                    #     ])
                    #     assert np_allvals_modif.shape[0] == adata_ct.obs['original_vs_modified'].value_counts()['modified'] * num_generated_realisations
    
                    #     plt.subplot(fig_num_rowcols, fig_num_rowcols, cnt_subfig); cnt_subfig += 1
                    #     sns.violinplot(
                    #         {'original':np_allvals_orig,
                    #          'perturbed':np_allvals_modif},
                    #         cut=0
                    #     )
                    #     plt.title("gene name: {}".format(g))
                        
    
                    # plt.suptitle("cell type: {}".format(ct), fontsize=24, y=0.95)
                    # plt.show()
    
                    
                    # # box/scatter plots 
                    # fig_num_rowcols = int(np.ceil(np.sqrt(
                    #     2 * len(adata_ct.uns['rank_genes_groups']['names'].tolist())
                    # )))
                    # plt.figure(figsize=[fig_num_rowcols*4, fig_num_rowcols*4])
                    # cnt_subfig = 1
                    # for g in [g1 for g1, _ in adata_ct.uns['rank_genes_groups']['names'].tolist()] +\
                    # [g2 for _, g2 in adata_ct.uns['rank_genes_groups']['names'].tolist()]:
                        
                    #     # get all original values
                    #     np_allvals_orig = np.concatenate([
                    #         xspl[list_selflag_orig]\
                    #          [:, adata_ct.var.index.tolist().index(g)].flatten() for xspl in  dict_original_varname_to_all_realisations['x_spl']
                    #     ])
                    #     assert np_allvals_orig.shape[0] == adata_ct.obs['original_vs_modified'].value_counts()['original'] * num_generated_realisations
    
                    #     # get all modified values
                    #     np_allvals_modif = np.concatenate([
                    #         xspl[list_selflag_modif]\
                    #          [:, adata_ct.var.index.tolist().index(g)].flatten() for xspl in  dict_modified_varname_to_all_realisations['x_spl']
                    #     ])
                    #     assert np_allvals_modif.shape[0] == adata_ct.obs['original_vs_modified'].value_counts()['modified'] * num_generated_realisations
    
                    #     plt.subplot(fig_num_rowcols, fig_num_rowcols, cnt_subfig); cnt_subfig += 1
                        
                    #     # sns.violinplot(
                    #     #     {'original':np_allvals_orig, 'perturbed':np_allvals_modif},
                    #     #     cut=0
                    #     # )
    
                    #     PROPS = {
                    #         'boxprops':{'facecolor':'none', 'edgecolor':'k'}
                    #     }
                    #     sns.boxplot(
                    #         {'original':np_allvals_orig,
                    #          'perturbed':np_allvals_modif},
                    #         **PROPS
                    #     )
                    #     plt.title("gene name: {}".format(g))
                        
                        
                        
                    #     for i in [1,2]:
                    #         y = [np_allvals_orig, np_allvals_modif][i-1]
                    #         # Add some random "jitter" to the x-axis
                    #         x = np.random.normal(i-1, 0.05, size=len(y))
                    #         plt.scatter(
                    #             x,
                    #             y,
                    #             alpha=1.0,
                    #             facecolors='none',
                    #             s=100.0,
                    #             edgecolor=['b', 'r'][i-1]
                    #         )
                        
    
                    # plt.suptitle("cell type: {}".format(ct), fontsize=24, y=0.95)
                    # plt.show()
                        
    
                    # assert False

In [ ]:
## Extract unperturbed dataset if not already loaded

adata_unperturbed = sc.read_h5ad("/nfs/team361/aa36/InflowCopiedFiles/Daniyal/inflow_DJ/adata_unperturbed.h5ad")
adata_unperturbed

In [ ]:
## Identify TLS cells in unperturbed data and assign labels as level 4

## Mapping of file names to desired level_4_cell_type labels
cell_id_files = {
    "CD8pos_TLS_border_cell_ids.txt": "TLS_CD8_border_unperturbed",
    "CD8pos_TLS_core_1_cell_ids.txt": "TLS_CD8_core_1_unperturbed",
    "CD8pos_TLS_core_2_cell_ids.txt": "TLS_CD8_core_2_unperturbed",
}

## Base directory where the files are located
base_path = "/nfs/users/nfs_d/dj17/inflow_DJ/"

## Step 1: Initialize level_4_cell_type from level_3_cell_type
adata_unperturbed.obs['level_4_cell_type'] = adata_unperturbed.obs['level_3_cell_type'].copy()

## Step 2: Loop through each file-label pair
for filename, new_label in cell_id_files.items():
    file_path = base_path + filename

    ## Load cell IDs from file
    with open(file_path, 'r') as f:
        cell_ids = [line.strip() for line in f.readlines()]

    ## Make sure the new label is allowed if the column is categorical
    if pd.api.types.is_categorical_dtype(adata_unperturbed.obs['level_4_cell_type']):
        if new_label not in adata_unperturbed.obs['level_4_cell_type'].cat.categories:
            adata_unperturbed.obs['level_4_cell_type'] = adata_unperturbed.obs['level_4_cell_type'].cat.add_categories([new_label])

    ## Filter to matching cell IDs
    matching_ids = [cid for cid in cell_ids if cid in adata_unperturbed.obs_names]

    ## Assign the new label
    adata_unperturbed.obs.loc[matching_ids, 'level_4_cell_type'] = new_label

## Optional: Sanity check
print(adata_unperturbed.obs['level_4_cell_type'].value_counts())


In [ ]:
# Define the target cell types to keep

target_unperturbed_types = [
    "TLS_CD8_border_unperturbed",
    "TLS_CD8_core_1_unperturbed",
    "TLS_CD8_core_2_unperturbed"
]

# Subset the unperturbed AnnData object
subset_unperturbed = adata_unperturbed[
    adata_unperturbed.obs['level_4_cell_type'].isin(target_unperturbed_types)
].copy()

# Quick sanity check
print(subset_unperturbed)
print(subset_unperturbed.obs['level_4_cell_type'].value_counts())


In [ ]:
## Extract perturbed dataset

adata_perturbed = sc.read_h5ad("/nfs/team361/aa36/InflowCopiedFiles/Daniyal/inflow_DJ/adata_perturbed.h5ad")
adata_perturbed

In [ ]:
## Identify TLS cells in perturbed data and assign labels as level 4

## Mapping of file names to desired level_4_cell_type labels
cell_id_files = {
    "CD8pos_TLS_border_cell_ids.txt": "TLS_CD8_border_perturbed",
    "CD8pos_TLS_core_1_cell_ids.txt": "TLS_CD8_core_1_perturbed",
    "CD8pos_TLS_core_2_cell_ids.txt": "TLS_CD8_core_2_perturbed",
}

## Base directory where the files are located
base_path = "/nfs/users/nfs_d/dj17/inflow_DJ/"

## Step 1: Initialize level_4_cell_type from level_3_cell_type
adata_perturbed.obs['level_4_cell_type'] = adata_perturbed.obs['level_3_cell_type'].copy()

## Step 2: Loop through each file-label pair
for filename, new_label in cell_id_files.items():
    file_path = base_path + filename

    ## Load cell IDs from file
    with open(file_path, 'r') as f:
        cell_ids = [line.strip() for line in f.readlines()]

    ## Make sure the new label is allowed if the column is categorical
    if pd.api.types.is_categorical_dtype(adata_perturbed.obs['level_4_cell_type']):
        if new_label not in adata_perturbed.obs['level_4_cell_type'].cat.categories:
            adata_perturbed.obs['level_4_cell_type'] = adata_perturbed.obs['level_4_cell_type'].cat.add_categories([new_label])

    ## Filter to matching cell IDs
    matching_ids = [cid for cid in cell_ids if cid in adata_perturbed.obs_names]

    ## Assign the new label
    adata_perturbed.obs.loc[matching_ids, 'level_4_cell_type'] = new_label

## Optional: Sanity check
print(adata_perturbed.obs['level_4_cell_type'].value_counts())


In [ ]:
## Define the target cell types to keep

target_perturbed_types = [
    "TLS_CD8_border_perturbed",
    "TLS_CD8_core_1_perturbed",
    "TLS_CD8_core_2_perturbed"
]

# Subset the unperturbed AnnData object
subset_perturbed = adata_perturbed[
    adata_perturbed.obs['level_4_cell_type'].isin(target_perturbed_types)
].copy()

# ✅ Quick sanity check
print(subset_perturbed)
print(subset_perturbed.obs['level_4_cell_type'].value_counts())


In [ ]:
## Merge AnnData objects

from anndata import concat

combined_subset = concat(
    [subset_unperturbed, subset_perturbed],
    join='outer',
    label='perturbation_state',
    keys=['unperturbed', 'perturbed'],
    index_unique=None
)

# Manually merge .uns content
combined_subset.uns = {}

# Optionally prefix keys to keep them distinct
for k, v in subset_unperturbed.uns.items():
    combined_subset.uns[f"unperturbed_{k}"] = v

for k, v in subset_perturbed.uns.items():
    combined_subset.uns[f"perturbed_{k}"] = v

print(combined_subset)

In [ ]:
## Differential expression

# Step 1: Normalize and log-transform the model-predicted counts
sc.pp.normalize_total(combined_subset, target_sum=1e4, layer='counts')
sc.pp.log1p(combined_subset, layer='counts')  # creates a temporary log-transformed layer used for DE

# Step 2: Run DE test using model-generated counts
sc.tl.rank_genes_groups(
    combined_subset,
    groupby='perturbation_state',     
    method='wilcoxon',                # or 't-test' or 'logreg'
    use_raw=False,
    layer='counts'                    # using model-predicted counts
)

# Step 3: Plot top differentially expressed genes
sc.pl.rank_genes_groups(
    combined_subset,
    n_genes=100,
    sharey=False,
    title="Differential Expression (Predicted Counts)"
)

# Step 4: Print top DE genes for each group
result = combined_subset.uns['rank_genes_groups']
groups = result['names'].dtype.names
for group in groups:
    print(f"Top genes in '{group}':")
    print(result['names'][group][:300])


In [ ]:
## Alternative differential expression for CD8+ T cell subsets

# Now run the differential expression analysis using these counts
sc.tl.rank_genes_groups(subset_perturbed, groupby='level_4_cell_type', method='wilcoxon')
sc.pl.rank_genes_groups(subset_perturbed, n_genes=100, sharey=False)

# Optionally, view the top genes per cluster
result = subset_perturbed.uns['rank_genes_groups']
groups = result['names'].dtype.names  # cluster names
for group in groups:
    print(f"Top genes in {group}:")
    print(result['names'][group][:100])

In [ ]:
# GENERATES FIGURE 7F

sc.pl.dotplot(
    combined_subset,
    var_names=['LILRB1', 'ZAP70', 'CD27', 'CD40LG'], #Change genes as required
    groupby='level_4_cell_type',
    layer='counts',           # model-predicted expression
    standard_scale='var',
    color_map='Reds',
    categories_order=[
        'TLS_CD8_core_1_unperturbed',
        'TLS_CD8_core_2_unperturbed',
        'TLS_CD8_border_unperturbed',
        'TLS_CD8_core_1_perturbed',
        'TLS_CD8_core_2_perturbed',
        'TLS_CD8_border_perturbed'
    ]
)